# Baseline only: copy the completed GRL run's settings
Enable GPU and Internet. Attach the original dataset and your saved GRL run outputs.
The selected run folder must contain `args.yaml` and `configuration.yaml`.
Source architecture/dataset YAML and original pretrained weights must remain accessible;
supply copies below if the old paths are unavailable. Do not use trained GRL weights.
This notebook runs no GRL training. It copies the resolved parameters, disables GRL,
and uses native Ultralytics output naming. Changed file paths are recorded for review.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import zipfile

DATASET_ROOT = "/kaggle/input/datasets/doquocbinhbinh/adaptiverdd-prepared"
# Set DATASET_ROOT="" only if you want automatic discovery.
PROBE_IMAGES = ["source/images/China_Drone_000000.jpg",
                "target_adaptation/images/China_MotorBike_000001.jpg"]
# Optional direct-read diagnostics; use [] for a dataset with different filenames.
REPO_URL = "https://github.com/binhdoquoc/GRL_RDD26.git"
REPO_REF = ""  # empty uses the repository default branch; or set a tag/commit
REPO_DIR = Path("/kaggle/working/GRL_RDD26")
PROJECT_DIR = REPO_DIR  # project files are at the repository root
NAMES = ["D00", "D10", "D20", "D40"]
TARGET_SPLIT = "target_adaptation"  # use "target" for a legacy dataset
VALIDATION_SPLIT = "target_eval"  # or a separate labeled source_eval split
TEST_SPLIT = "target_test"       # set "" to disable final test evaluation
RUN_TESTS = True
RUN_BASELINE = False
TORCH_INDEX_URL = "https://pypi.org/simple"  # avoids the timed-out download.pytorch.org host
RESUME = ""  # trusted native weights/resume.pt from an interrupted run

In [ ]:
GRL_RUN_DIR = "/kaggle/input/YOUR-SAVED-GRL-RUN/train"
# In the same live session, use the actual completed runs/detect/train folder.
ORIGINAL_DATA_YAML = ""  # optional copy of original kaggle_data.yaml
ORIGINAL_MODEL_YAML = ""  # optional copy of original architecture YAML
# Leave empty when original paths still exist. Preserve original YAML contents.

## 1. Clone the project
Reruns reuse the checkout and preserve edited YAML and results. To update code, start a fresh session or explicitly update the checkout before setup.


In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    if REPO_REF:
        subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
elif not (REPO_DIR / ".git").is_dir():
    raise RuntimeError(f"Existing path is not a Git checkout: {REPO_DIR}")
if REPO_REF:
    wanted = subprocess.check_output(["git", "rev-parse", REPO_REF], cwd=REPO_DIR, text=True).strip()
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
    if actual != wanted:
        raise RuntimeError("Existing checkout differs from REPO_REF. Start a fresh session or update it explicitly.")
if not (PROJECT_DIR / "train.py").is_file():
    raise FileNotFoundError("The cloned revision lacks train.py at the repository root. Upload the project contents, then select that revision.")
print("Project:", PROJECT_DIR)
print("Revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip())
PROJECT_FILES = [str(p.relative_to(PROJECT_DIR))
                 for folder in ("grl_yolo", "configs", "scripts", "tests")
                 for p in (PROJECT_DIR / folder).rglob("*")
                 if p.is_file() and p.suffix in {".py", ".yaml"}]
PROJECT_FILES += ["train.py", "requirements.txt", "pyproject.toml", "pytest.ini", "README.md"]

## Optional direct file-read diagnostics
These probes print the actual file-open result; they do not stop training on a
directory metadata check. The supplied paths are passed directly to Ultralytics.

In [ ]:
sys.path.insert(0, str(PROJECT_DIR))
import importlib
import grl_yolo.kaggle as kaggle_data
importlib.reload(kaggle_data)
root = Path(DATASET_ROOT) if DATASET_ROOT else kaggle_data.discover_dataset()
for relative in PROBE_IMAGES:
    sample = root / relative
    try:
        with sample.open("rb") as stream:
            header = stream.read(16)
        print("Image read OK:", sample, "bytes checked:", len(header))
    except OSError as error:
        print("Image read FAILED:", sample, type(error).__name__, str(error))
print("Training dataset:", root)
print("File probes are diagnostic. No custom directory-access gate is applied.")

## 2. Set up the isolated Python environment
The notebook kernel may use Python 3.13. Training uses a separate **Python 3.12**
interpreter matching the pinned project dependencies. Standalone `uv` creates it
without invoking the kernel's broken/missing `ensurepip`; no kernel pip is required.
Python and package downloads need Internet. PyTorch is installed from **PyPI**
by default to avoid timeouts reaching `download.pytorch.org`. Its Linux wheel
includes CUDA dependencies; CUDA availability is checked after installation.
Training subprocesses use the `Agg` Matplotlib backend to avoid inheriting
the notebook kernel's unavailable inline backend. Saved plots display below.
The kernel does not need restarting.

In [ ]:
sys.path.insert(0, str(PROJECT_DIR))
import importlib
import grl_yolo.environment as training_environment
# Reload restored code if this notebook is imported into an existing session.
importlib.reload(training_environment)
print("Notebook kernel:", sys.version)
PYTHON, ENV = training_environment.prepare_environment(PROJECT_DIR, torch_index=TORCH_INDEX_URL)

from grl_yolo.process import run_logged
COMMAND_NUMBER = 0

def run_project(arguments):
    global COMMAND_NUMBER
    COMMAND_NUMBER += 1
    filename = "train.log" if str(arguments[0]) == "train.py" else f"command-{COMMAND_NUMBER:02d}.log"
    return run_logged([str(PYTHON), *map(str, arguments)], cwd=PROJECT_DIR, env=ENV,
                      log_file=PROJECT_DIR / "logs" / filename)

runtime = subprocess.run(
    [str(PYTHON), "-c", "import sys; print('Training Python:', sys.version, flush=True); "
     "import torch, ultralytics; print(torch.__version__, ultralytics.__version__, flush=True); "
     "print('CUDA runtime:', torch.version.cuda, 'CUDA available:', torch.cuda.is_available(), flush=True); "
     "assert torch.cuda.is_available(), 'Enable a GPU in Kaggle settings'; "
     "print(torch.cuda.get_device_name(0), flush=True)"],
    cwd=PROJECT_DIR, env=ENV, text=True, capture_output=True,
)
print(runtime.stdout)
if runtime.stderr:
    print(runtime.stderr)
if runtime.returncode:
    raise RuntimeError("Training runtime check failed. The underlying error is printed above.")
if RUN_TESTS:
    subprocess.run([str(PYTHON), "-m", "pytest", "-q"], cwd=PROJECT_DIR,
                   env={**ENV, "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2"}, check=True)

## Run baseline only
Settings come from the saved run, not the notebook's default training template.
The original dataset YAML is retained; no default dataset YAML is generated.

In [ ]:
command = ["scripts/run_baseline.py", "--grl-run", GRL_RUN_DIR]
if ORIGINAL_DATA_YAML:
    command += ["--data", ORIGINAL_DATA_YAML]
if ORIGINAL_MODEL_YAML:
    command += ["--model", ORIGINAL_MODEL_YAML]
run_project(command)
BASELINE_DIR = Path(json.loads((PROJECT_DIR / "notebook_baseline.json").read_text())["run_dir"])
print("Baseline results:", BASELINE_DIR)
print("Parameter copy record:", PROJECT_DIR / "baseline_settings_copy.json")

## Evaluate the best baseline checkpoint on the original test split

In [ ]:
evaluation = 'import sys, yaml\nfrom ultralytics import YOLO\nweights, config = sys.argv[1:]\nsettings = yaml.safe_load(open(config))["train"]\nmetrics = YOLO(weights).val(data=settings["data"], split="test", device=settings["device"],\n                          imgsz=settings["imgsz"], batch=settings["batch"])\nprint(metrics.results_dict)\nprint("Evaluation outputs:", metrics.save_dir)\n'

run_project(["-c", evaluation, BASELINE_DIR / "weights/best_detector.pt",
             PROJECT_DIR / "configs/baseline_from_grl.yaml"])